# Chương 3: FKG-UM / FKG-MM trên BRSET 16.266 ảnh (5 fold theo bệnh nhân)

Bộ mã `FKGMM_BRSET_Chuong3`, **không cần GPU** (Runtime CPU là đủ).

| Bước | Lệnh | Kết quả |
|---|---|---|
| Dựng 7 nhóm dữ liệu + M1–M6, EPS với FISA-GM-w | `chuong3.py all` | `chuong3/ket_qua/ket_qua.md`, `chuong3/ket_qua/hinh/` (ROC, PR, AP theo fold) |
| Lọc luật: trùng giữ 1, mâu thuẫn giữ luật có độ tin cậy cao hơn | `chuong3.py run --rule-base dedup_conf` | `chuong3_dedup/ket_qua/ket_qua.md` |
| Kiểm toán FISA-GM / GM-w (tuỳ chọn) | `chuong3.py audit` | `chuong3/ket_qua_kiem_toan/` |

**Chuẩn bị:** tải `FKGMM_BRSET_Chuong3.zip` lên `MyDrive/`, chạy lần lượt các ô. Kết quả và log nằm trên Drive;
phiên bị ngắt thì chạy lại từ đầu, phần đã xong được bỏ qua. Thời gian tham khảo trên máy 8 nhân:
dữ liệu khoảng 5 phút, mỗi lượt kịch bản khoảng 35 phút, kiểm toán khoảng 30 phút (Colab 2 nhân có thể lâu hơn).

In [ ]:
from pathlib import Path
MODE = "full"                      # "full" | "smoke" (dữ liệu tổng hợp, vài phút, KHÔNG phải kết quả BRSET)
ZIP_PATH = Path("/content/drive/MyDrive/FKGMM_BRSET_Chuong3.zip")
OUTPUT_ROOT = Path("/content/drive/MyDrive/FKGE_BRSET_chuong3")
RUN_MAIN, RUN_DEDUP, RUN_AUDIT = True, True, False
WORKDIR = Path("/content")

## 1. Gắn Drive, giải nén, cài thư viện, chạy test Chương 3

In [ ]:
import hashlib, json, subprocess, sys, time, zipfile
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Không chạy trên Colab: bỏ qua bước gắn Drive")
SMOKE = MODE == "smoke"
if SMOKE:
    OUTPUT_ROOT = OUTPUT_ROOT.with_name(OUTPUT_ROOT.name + "_SMOKE")
assert ZIP_PATH.exists(), f"Chưa thấy {ZIP_PATH}: tải FKGMM_BRSET_Chuong3.zip lên Drive"
with zipfile.ZipFile(ZIP_PATH) as z:
    assert all((WORKDIR / n).resolve().is_relative_to(WORKDIR.resolve()) for n in z.namelist()), "Đường dẫn lạ trong zip"
    z.extractall(WORKDIR)
bundle = WORKDIR / "FKGMM_BRSET_Chuong3"
print("ZIP sha256:", hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
OUT, OUT_DEDUP, LOGS = OUTPUT_ROOT / "chuong3", OUTPUT_ROOT / "chuong3_dedup", OUTPUT_ROOT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

def run_logged(args, log_name):
    """Chạy lệnh trong gói, in log ra notebook và ghi đầy đủ lên Drive; lỗi thì dừng ô."""
    command = [sys.executable, "-B", "-X", "utf8", "-u", *map(str, args)]
    print("Lệnh:", " ".join(command), flush=True)
    with open(LOGS / log_name, "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} {' '.join(command)}\n")
        process = subprocess.Popen(command, cwd=bundle, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        for line in process.stdout:
            log.write(line)
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"Lệnh lỗi (mã {code}); xem {LOGS / log_name}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(bundle / "requirements.txt")], check=True)
run_logged(["-m", "unittest", "tests.test_ch3"], "tests.log")

## 2. Dữ liệu 7 nhóm và kịch bản chính (FISA-GM-w)

Dựng `table`, `image`, `fusion` (Φsel), `fusion_filter`, `fusion_hadamard`, `fusion_tensor` (Thuật toán 3.5, ρ = 6),
`fusion_wrapper` cho 5 fold; mọi bước fit chỉ trên tập huấn luyện của fold. Sau đó chạy M1–M6 và EPS.

In [ ]:
if RUN_MAIN:
    run_logged(["chuong3.py", "all", "--out", OUT, "--resume", *(["--smoke"] if SMOKE else [])], "chuong3.log")

## 3. Biến thể lọc luật trên cùng dữ liệu

Trên cơ sở luật huấn luyện của mỗi fold (gồm SMOTE): luật trùng giữ 1; với tiền đề mâu thuẫn, giữ nhãn có độ tin cậy
conf(A → l) cao hơn, bỏ nhãn kia; hai nhãn hoà thì bỏ cả hai. Trọng số GM-w giữ nguyên cách ước lượng.

In [ ]:
if RUN_DEDUP:
    run_logged(["chuong3.py", "run", "--out", OUT_DEDUP, "--data", OUT / "data", "--rule-base", "dedup_conf", "--resume"],
               "chuong3_dedup.log")

## 4. Kiểm toán FISA-GM / GM-w (tuỳ chọn, đặt `RUN_AUDIT = True`)

In [ ]:
if RUN_AUDIT:
    run_logged(["chuong3.py", "audit", "--out", OUT], "chuong3_audit.log")

## 5. Xem kết quả

In [ ]:
from IPython.display import Image, Markdown, display
for title, path in (("FISA-GM-w, cơ sở luật đầy đủ", OUT / "ket_qua/ket_qua.md"),
                    ("FISA-GM-w, lọc luật trùng/mâu thuẫn", OUT_DEDUP / "ket_qua/ket_qua.md"),
                    ("Kiểm toán", OUT / "ket_qua_kiem_toan/ket_qua_kiem_toan.md")):
    if path.exists():
        display(Markdown(f"---\n## {title}\n\n" + path.read_text(encoding="utf-8")))
for name in ("roc_pr_kb_m1_m2.png", "roc_pr_kb_m5.png", "ap_theo_fold.png"):
    for base in (OUT, OUT_DEDUP):
        path = base / "ket_qua/hinh" / name
        if path.exists():
            display(Markdown(f"**{base.name}: {name}**"))
            display(Image(filename=str(path)))


## 6. Đóng gói để gửi lại

In [ ]:
stamp = time.strftime("%Y%m%d_%H%M")
package = OUTPUT_ROOT.parent / f"{OUTPUT_ROOT.name}_bao_cao_{stamp}.zip"
paths = [*LOGS.rglob("*"), OUT / "data/prepare_manifest.json"]
for base in (OUT, OUT_DEDUP):
    paths += [*(base / "ket_qua").rglob("*"), *(base / "ket_qua_kiem_toan").rglob("*")]   # gồm hinh/ và predictions/
paths += [*(OUT / "data").glob("*/fold_*/metadata.json"), *(OUT / "data").glob("*/fold_*/selected_features.csv")]
with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as z:
    for path in paths:
        if path.is_file():
            z.write(path, path.relative_to(OUTPUT_ROOT))
print("Gửi file:", package, f"({package.stat().st_size / 2**20:.1f} MiB)")